# GPUBMA — exhaustive 2^30 enumeration on an NVIDIA A100 (Google Colab)

Runs the **validated** GPUBMA exhaustive enumerator (streamed direct
batching, float64 only, deterministic reductions, checkpoint/resume — see
`docs/ADR_0001_GPU_ENUMERATOR.md` and `docs/FWL_BLOCK_FORMULATION.md`) on
the frozen `panel_30` dataset: **p = 30 optional predictors,
2^30 = 1,073,741,824 models**, shrink (Stata-verified) convention,
controls `w1 w2`, g = 1000, beta-binomial(1, 1) model prior — followed by
exact BMA postestimation graphics (PMP, model size, variable-inclusion
map, coefficient densities, PIP).

**How to run**
1. Open this notebook in Colab (badge in the repository README, or
   `File → Open notebook → GitHub`).
2. Runtime → Change runtime type → **A100 GPU**.
3. Run all. The repository is cloned from GitHub automatically (edit
   `GITHUB_REPO_URL` in the first code cell if you use a fork; a copy of
   the repository placed at `MyDrive/GPUBMA` takes precedence).

**Cost (Measured on an A100-SXM4-40GB)**
- full 2^30 enumeration: ~95 s;
- optional second streaming pass (`[PASS2]`, exact coefficient densities
  and top-K extraction): ~10–12 minutes.

Checkpoints are written to Drive every 60 s, so a Colab disconnect loses
at most ~1 minute of work: reopen and **Run all** — the run resumes from
the checkpoint automatically, and finished stages are skipped (idempotent).
All results, figures (PNG/SVG/PDF), and tables (CSV/Parquet) persist under
`MyDrive/GPUBMA_p30_run/`.


In [ ]:
# [COLAB] Google Drive (persistent checkpoints/results) + gpubma source
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

# Checkpoints, results, figures, and tables persist here across Colab
# disconnects. Created automatically.
WORK_DIR = Path('/content/drive/MyDrive/GPUBMA_p30_run')
WORK_DIR.mkdir(parents=True, exist_ok=True)

# gpubma source code: a Drive copy at MyDrive/GPUBMA is used when present;
# otherwise the repository is cloned from GitHub (edit the URL if you use
# a fork).
GITHUB_REPO_URL = 'https://github.com/Favioleiva/gpubma.git'
DRIVE_REPO = Path('/content/drive/MyDrive/GPUBMA')
if (DRIVE_REPO / 'pyproject.toml').exists():
    REPO_DIR = DRIVE_REPO
else:
    REPO_DIR = Path('/content/gpubma')
    if not (REPO_DIR / 'pyproject.toml').exists():
        !git clone --depth 1 "$GITHUB_REPO_URL" "$REPO_DIR"
assert (REPO_DIR / 'pyproject.toml').exists(), (
    'gpubma source not found: copy the repository to Drive at '
    f'{DRIVE_REPO} or set GITHUB_REPO_URL to a reachable repository')
print('repo :', REPO_DIR)
print('work :', WORK_DIR)


In [ ]:
# [COLAB] Install gpubma from the repository copy (plus psutil for host memory)
%pip install --quiet $REPO_DIR/. psutil

import importlib.metadata, subprocess
GPUBMA_VERSION = importlib.metadata.version('gpubma')
try:
    GIT_COMMIT = subprocess.run(
        ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
        capture_output=True, text=True, check=True).stdout.strip()
except Exception:
    GIT_COMMIT = 'unknown (repository copied without .git)'
print('gpubma', GPUBMA_VERSION, '@', GIT_COMMIT)


In [ ]:
# [COLAB] Verify CUDA, GPU name, VRAM, compute capability, and real float64
import torch

assert torch.cuda.is_available(), 'CUDA is not available — select a GPU runtime'
props = torch.cuda.get_device_properties(0)
free_b, total_b = torch.cuda.mem_get_info()
GPU_NAME = props.name
print(f'GPU: {GPU_NAME}  CC {props.major}.{props.minor}')
print(f'VRAM: total {total_b / 2**30:.1f} GiB, free {free_b / 2**30:.1f} GiB')
print(f'torch {torch.__version__} (CUDA {torch.version.cuda})')

# genuine float64 execution check (not just a capability flag)
a = torch.randn(256, 256, dtype=torch.float64, device='cuda')
r = (a @ a.T).sum()
torch.cuda.synchronize()
assert r.dtype == torch.float64
ref = (a.cpu().numpy() @ a.cpu().numpy().T).sum()
assert abs(float(r) - float(ref)) < 1e-6, 'float64 GPU result inconsistent'
print('float64 GPU execution: OK')

# CuPy is NOT used by gpubma (backend is torch); reported for completeness
try:
    import cupy
    print('cupy present (informational, unused):', cupy.__version__)
except ImportError:
    print('cupy not installed (fine — gpubma uses torch)')

ALLOW_NON_A100 = False  # set True only if you deliberately use another GPU
if 'A100' not in GPU_NAME and not ALLOW_NON_A100:
    raise RuntimeError(f'expected an A100, got {GPU_NAME}; '
                       'set ALLOW_NON_A100 = True to override')


In [ ]:
# [COLAB] gpubma doctor (hardware diagnostic, saved next to the results)
!python -m gpubma.doctor --json "$WORK_DIR/gpu_doctor_colab.json"


In [ ]:
# [COLAB] Run configuration (validated statistical setup; conservative VRAM)
P = 30                      # 2^30 = 1,073,741,824 models
TOP_K = 20
CHECKPOINT_EVERY_S = 60.0   # Drive checkpoint cadence (disconnect-safe)
PROGRESS_EVERY_S = 30.0

DATA_PARQUET = REPO_DIR / 'data' / 'synthetic' / 'panel_30.parquet'
META_JSON = REPO_DIR / 'data' / 'synthetic' / 'panel_30_metadata.json'
CKPT_PATH = WORK_DIR / f'enum_p{P}.ckpt.npz'
OUT_DIR = WORK_DIR / f'results_p{P}'
OUT_DIR.mkdir(exist_ok=True)

# conservative automatic batch sizing from ACTUAL free VRAM (A100 40 GB and
# 80 GB differ): ~25% of free memory as working budget, chunk cap scaled.
free_b, total_b = torch.cuda.mem_get_info()
VRAM_BUDGET = int(0.25 * free_b)
MAX_CHUNK = 1 << 17 if free_b > 30 * 2**30 else 1 << 16
print(f'VRAM budget {VRAM_BUDGET / 2**30:.1f} GiB, max chunk {MAX_CHUNK:,}')


In [ ]:
# [CORE] Load frozen panel_30, verify checksum, prepare validated inputs
import hashlib, json
import numpy as np
import pandas as pd

meta = json.loads(Path(META_JSON).read_text())
sha = hashlib.sha256(Path(DATA_PARQUET).read_bytes()).hexdigest()
assert sha == meta['file_checksums']['parquet']['sha256'], (
    'panel_30.parquet checksum mismatch — dataset is not the frozen artifact')
print('panel_30.parquet sha256 verified:', sha[:16], '...')

df = pd.read_parquet(DATA_PARQUET)
n = len(df)
assert n == 1000 and meta['expected_number_of_models'] == 1 << 30
predictors = [f'x{j}' for j in range(1, P + 1)]

# shrink (Stata-verified) convention: residualize on [1, w1, w2],
# df = n - 1, TSS_c normalizer, joint g-prior over optional + always slopes
y = df['y'].to_numpy(np.float64)
X = df[predictors].to_numpy(np.float64)
A = np.column_stack([np.ones(n), df[['w1', 'w2']].to_numpy(np.float64)])
Q, _ = np.linalg.qr(A)
y_r = y - Q @ (Q.T @ y)
X_r = X - Q @ (Q.T @ X)
yc = y - y.mean()
CONV = dict(df_resid=n - 1, tss_norm=float(yc @ yc), k_always=2)
G = float(max(n, P * P))  # = 1000 for n = 1000, P <= 31 — validated default

from gpubma.priors.model_priors import log_model_prior_function
LOG_PRIOR, prior_desc = log_model_prior_function(('betabinomial', 1.0, 1.0), P)
N_EXPECTED = 1 << P
print(f'p = {P}, models = {N_EXPECTED:,}, g = {G:.0f}, prior = {prior_desc}')


In [ ]:
# [CORE] Enumerate — resumes from Drive checkpoint; skips if already complete
import time
import gpubma.gpu.enumerator as _en
from gpubma.gpu.enumerator import enumerate_models_gpu

RESULTS_JSON = OUT_DIR / 'results.json'
result = None
if RESULTS_JSON.exists():
    print('final results already on Drive — skipping enumeration (idempotent)')
else:
    # count checkpoint writes for the benchmark report
    _orig_save = _en._save_checkpoint
    CKPT_WRITES = [0]
    def _counting_save(path, state):
        _orig_save(path, state)
        CKPT_WRITES[0] += 1
    _en._save_checkpoint = _counting_save

    t_wall = time.time()
    def show_progress(info):
        rate = info['models_per_second']
        remaining = (info['models_total'] - info['models_done']) / max(rate, 1e-9)
        print(f"[{info['fraction']:7.2%}] "
              f"{info['models_done']:,}/{info['models_total']:,} models  "
              f"k={info['current_size']:2d}  elapsed {info['elapsed_s']:,.0f} s  "
              f"{rate:,.0f} models/s  ETA {remaining:,.0f} s  "
              f"GPU {torch.cuda.memory_allocated() / 2**30:.2f} GiB "
              f"(peak {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB)",
              flush=True)

    resume = CKPT_PATH.exists()
    print('resuming from Drive checkpoint' if resume else 'starting fresh')
    result = enumerate_models_gpu(
        X_r, y_r, g=G, log_model_prior=LOG_PRIOR, top_k=TOP_K,
        vram_budget_bytes=VRAM_BUDGET, max_chunk=MAX_CHUNK,
        checkpoint_path=CKPT_PATH, checkpoint_every_s=CHECKPOINT_EVERY_S,
        resume=resume, progress_every_s=PROGRESS_EVERY_S,
        progress=show_progress, **CONV)
    WALL_S = time.time() - t_wall
    N_CKPT = CKPT_WRITES[0]
    _en._save_checkpoint = _orig_save
    print(f"done: {result['n_models_evaluated']:,} models in "
          f"{result['runtime']['elapsed_s']:,.1f} s "
          f"({result['runtime']['models_per_second']:,.0f} models/s)")


In [ ]:
# [CORE] Exact-count assertion and posterior validation
if result is not None:
    assert result['n_models_evaluated'] == N_EXPECTED == (1 << P), (
        f"processed {result['n_models_evaluated']:,} != expected {N_EXPECTED:,}")
    if P == 30:
        assert result['n_models_evaluated'] == 1_073_741_824
    sd_sum = result['normalization_check']['size_distribution_sum']
    assert abs(sd_sum - 1.0) < 1e-9, f'size distribution sums to {sd_sum!r}'
    assert result['normalization_check']['pip_max_overshoot'] <= 1e-12
    assert (result['pip'] >= 0.0).all() and (result['pip'] <= 1.0).all()
    assert np.isfinite(result['log_normalizer'])
    assert result['runtime']['precision'] == 'float64'
    ms = float(np.arange(P + 1) @ result['size_distribution'])
    assert abs(ms - result['mean_model_size']) < 1e-9
    print(f"VALIDATED: exactly {result['n_models_evaluated']:,} models; "
          f"size-dist sum = {sd_sum:.15f}; PIPs in [0, 1]; "
          f"mean model size = {result['mean_model_size']:.6f}")
else:
    print('nothing to validate here — results were loaded below')


In [ ]:
# [CORE] Save results (JSON + CSV + Parquet) and the benchmark report
import platform, resource

def _results_payload(r):
    return {
        'p': P, 'g': G, 'model_prior': 'betabinomial(1,1)',
        'convention': 'shrink (Stata-verified)', 'precision': 'float64',
        'n_models_expected': int(N_EXPECTED),
        'n_models_evaluated': int(r['n_models_evaluated']),
        'log_normalizer': r['log_normalizer'],
        'mean_model_size': r['mean_model_size'],
        'pip': r['pip'].tolist(),
        'coef_mean': r['coef_mean'].tolist(),
        'coef_sd': r['coef_sd'].tolist(),
        'size_distribution': r['size_distribution'].tolist(),
        'top_models': r['top_models'],
        'normalization_check': r['normalization_check'],
        'runtime': r['runtime'],
    }

if result is not None:
    RESULTS_JSON.write_text(json.dumps(_results_payload(result), indent=2))
    payload = json.loads(RESULTS_JSON.read_text())
else:
    payload = json.loads(RESULTS_JSON.read_text())
    print('loaded previously saved results from Drive')

pred_tbl = pd.DataFrame({
    'predictor': predictors,
    'pip': payload['pip'],
    'posterior_mean': payload['coef_mean'],
    'posterior_sd': payload['coef_sd'],
})
size_tbl = pd.DataFrame({'model_size': range(P + 1),
                         'posterior_probability': payload['size_distribution']})
top_tbl = pd.DataFrame(payload['top_models'])
for name, tbl in [('predictors', pred_tbl), ('size_distribution', size_tbl),
                  ('top_models', top_tbl)]:
    tbl.to_csv(OUT_DIR / f'{name}.csv', index=False)
    tbl.to_parquet(OUT_DIR / f'{name}.parquet', index=False)

if result is not None:
    bench = {
        'label': 'Measured',
        'gpu': GPU_NAME,
        'runtime_s': payload['runtime']['elapsed_s'],
        'wall_clock_this_session_s': WALL_S,
        'models_per_second': payload['runtime']['models_per_second'],
        'peak_gpu_memory_bytes': payload['runtime']['peak_gpu_memory_bytes'],
        'peak_host_memory_kb_ru_maxrss':
            resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,
        'checkpoint_writes_this_session': N_CKPT,
        'chunks': payload['runtime']['chunks'],
        'resumed': payload['runtime']['resumed'],
        'gpubma_version': GPUBMA_VERSION,
        'git_commit': GIT_COMMIT,
        'python': platform.python_version(),
        'torch': torch.__version__,
        'n_models_evaluated': payload['n_models_evaluated'],
        'validation': {
            'exact_count_ok': payload['n_models_evaluated'] == N_EXPECTED,
            **payload['normalization_check'],
        },
    }
    (OUT_DIR / 'benchmark_report.json').write_text(json.dumps(bench, indent=2))
print('saved to', OUT_DIR)
sorted(p.name for p in OUT_DIR.iterdir())


In [ ]:
# [CORE] Compact results table (paste into STATUS.md)
rt = payload['runtime']
print('| p | models | elapsed s | models/s | peak GPU GiB | device | '
      'size-dist sum | exact count |')
print('|---|---|---|---|---|---|---|---|')
print(f"| {P} | {payload['n_models_evaluated']:,} "
      f"| {rt['elapsed_s']:,.1f} | {rt['models_per_second']:,.0f} "
      f"| {rt['peak_gpu_memory_bytes'] / 2**30:.2f} | {rt['device']} "
      f"| {payload['normalization_check']['size_distribution_sum']:.15f} "
      f"| {'yes' if payload['n_models_evaluated'] == N_EXPECTED else 'NO'} |")
print()
print('top 5 models (mask, size, pmp):')
for m in payload['top_models'][:5]:
    print(f"  {m['mask']:>10d}  size {m['size']:2d}  pmp {m['pmp']:.6f}")


## BMA postestimation graphics (exact, from the complete 2^30 posterior)

Python equivalents of Stata's `bmagraph pmp`, `bmagraph msize`,
`bmagraph varmap`, `bmagraph coefdensity`, plus a PIP chart — all built on
the EXACT exhaustive posterior. Model probabilities are always the global
PMPs normalized over all 2^30 models (never silently renormalized over the
displayed subset); every figure includes the exact omitted mass as
"Other models".

**Second streaming pass — required.** `results.json` stores only BMA-level
moments and the top 20 models. The coefficient-density mixture and any
top-K/coverage selection beyond 20 need model-level quantities, so the
`[PASS2]` cell runs a bounded-memory, resumable, deterministic float64
sweep that (a) extracts the top `K_MAX` models and (b) accumulates the
exact conditional coefficient densities (mixtures of Student-t densities)
on a fixed grid. It reuses the completed run's log-normalizer and
sufficient statistics — the primary enumeration is never repeated and its
results are never altered. These cells are independently rerunnable: they
load everything they need from Drive and skip finished work.


In [ ]:
# [PLOT-SETUP] Figure/table output helpers and saved-results loading
import json
import math
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

FIG_DIR = WORK_DIR / 'results' / f'p{P}' / 'figures'
PLOT_DATA_DIR = WORK_DIR / 'results' / f'p{P}' / 'plot_data'
FIG_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DATA_DIR.mkdir(parents=True, exist_ok=True)

if 'payload' not in dir():
    payload = json.loads((OUT_DIR / 'results.json').read_text())
    print('loaded saved p =', P, 'results from Drive (enumeration NOT rerun)')

PIP = np.asarray(payload['pip'])
COEF_MEAN = np.asarray(payload['coef_mean'])
COEF_SD = np.asarray(payload['coef_sd'])
SIZE_DIST = np.asarray(payload['size_distribution'])
LOG_NORMALIZER = float(payload['log_normalizer'])
N_MODELS = int(payload['n_models_evaluated'])
PRIOR_PIP = 0.5  # beta-binomial(1,1): prior inclusion probability a/(a+b)

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300,
                     'axes.grid': True, 'grid.alpha': 0.3,
                     'font.size': 10, 'axes.titlesize': 12})

def save_fig(fig, name):
    for ext in ('png', 'svg', 'pdf'):
        fig.savefig(FIG_DIR / f'{name}.{ext}', bbox_inches='tight')
    print('saved figure:', name, '(png/svg/pdf)')

def save_table(tbl, name):
    tbl.to_csv(PLOT_DATA_DIR / f'{name}.csv', index=False)
    tbl.to_parquet(PLOT_DATA_DIR / f'{name}.parquet', index=False)
    print('saved plot data:', name, '(csv/parquet)')

def mask_vars(mask):
    return [predictors[j] for j in range(P) if (int(mask) >> j) & 1]


In [ ]:
# [PASS2] Bounded-memory streaming pass: top-K models + EXACT conditional
# coefficient densities. REQUIRED because results.json lacks model-level
# conditional moments. Resumable (Drive checkpoint, SHA-256-gated),
# deterministic (fixed chunk order, one-hot einsum reductions, no atomics),
# float64 only. Reuses the saved log-normalizer: weights are exact global
# PMPs, and the completed primary enumeration is neither altered nor rerun.
import hashlib
import time
import torch
from gpubma.gpu.enumerator import binomial_table, unrank_combinations, \
    _save_checkpoint as _p2_save, _load_checkpoint as _p2_load

K_MAX = 4096         # top-K models to extract (covers 20/50/100 + coverage)
GRID_POINTS = 257    # coefficient-density grid resolution per predictor
PASS2_CHUNK = 8192   # bounded-memory chunk (density buffers dominate)
PASS2_CKPT = WORK_DIR / f'pass2_p{P}.ckpt.npz'
TOPK_PARQUET = OUT_DIR / 'pass2_topk_models.parquet'
DENSITY_NPZ = OUT_DIR / 'pass2_density_grids.npz'

SECOND_PASS_REQUIRED = True
print('coefdensity second pass required:', SECOND_PASS_REQUIRED,
      '(results.json has BMA-level moments + top 20 only)')

def _pass2_grid():
    # conditional-on-inclusion mean/sd from the saved BMA moments
    pip_safe = np.maximum(PIP, 1e-300)
    cm = COEF_MEAN / pip_safe
    cv = np.maximum((COEF_SD**2 + COEF_MEAN**2) / pip_safe - cm**2, 1e-30)
    cs = np.sqrt(cv)
    lo, hi = cm - 8.0 * cs, cm + 8.0 * cs
    return np.linspace(lo, hi, GRID_POINTS, axis=1)  # (P, G)

def run_pass2():
    dev = torch.device('cuda')
    n = len(y_r)
    Zxx_np = X_r.T @ X_r
    Zxy_np = X_r.T @ y_r
    tss = float(y_r @ y_r)
    tss_norm, k_alw, df = CONV['tss_norm'], CONV['k_always'], float(CONV['df_resid'])
    ess_always = tss_norm - tss
    lp = np.array([LOG_PRIOR(k) for k in range(P + 1)])
    grid_np = _pass2_grid()
    h = hashlib.sha256()
    for arr in (Zxx_np, Zxy_np, lp, grid_np):
        h.update(np.ascontiguousarray(arr, np.float64).tobytes())
    h.update(np.array([tss, tss_norm, df, G, k_alw, LOG_NORMALIZER,
                       K_MAX, GRID_POINTS], np.float64).tobytes())
    digest = h.hexdigest()

    Zxx = torch.from_numpy(Zxx_np).to(dev)
    Zxy = torch.from_numpy(Zxy_np).to(dev)
    grid = torch.from_numpy(grid_np).to(dev)          # (P, G)
    binom_np = binomial_table(P)
    binom = torch.from_numpy(binom_np).to(dev)
    s = G / (1.0 + G)
    log1pg = math.log1p(G)
    tiny = float(np.finfo(np.float64).tiny)
    c_t = (math.lgamma((df + 1) / 2) - math.lgamma(df / 2)
           - 0.5 * math.log(df * math.pi))

    dens = torch.zeros(P, GRID_POINTS, dtype=torch.float64, device=dev)
    top_s = torch.full((K_MAX,), -math.inf, dtype=torch.float64, device=dev)
    top_m = torch.zeros(K_MAX, dtype=torch.int64, device=dev)
    sum_w = torch.zeros((), dtype=torch.float64, device=dev)
    done = 0
    k0, r0 = 0, 0
    if PASS2_CKPT.exists():
        st = _p2_load(PASS2_CKPT)
        assert str(st['digest']) == digest, 'pass2 checkpoint config mismatch'
        dens = torch.from_numpy(st['dens']).to(dev)
        top_s = torch.from_numpy(st['top_s']).to(dev)
        top_m = torch.from_numpy(st['top_m']).to(dev)
        sum_w = torch.tensor(float(st['sum_w']), dtype=torch.float64, device=dev)
        done, k0, r0 = int(st['done']), int(st['next_k']), int(st['next_rank'])
        print(f'pass2: resuming at k={k0}, rank={r0:,} ({done:,} models done)')

    def save_ckpt(nk, nr):
        _p2_save(PASS2_CKPT, dict(
            digest=np.str_(digest), dens=dens.cpu().numpy(),
            top_s=top_s.cpu().numpy(), top_m=top_m.cpu().numpy(),
            sum_w=sum_w.cpu().numpy(), done=np.int64(done),
            next_k=np.int64(nk), next_rank=np.int64(nr)))

    t0, last_ck, last_pr = time.time(), time.time(), time.time()
    for k in range(k0, P + 1):
        total_k = int(binom_np[P, k])
        r = r0 if k == k0 else 0
        while r < total_k:
            B = min(PASS2_CHUNK, total_k - r)
            if k == 0:
                omr = max(tss / tss_norm, tiny)
                sc = torch.tensor([0.5 * (df - k_alw) * log1pg
                                   - 0.5 * df * math.log1p(G * omr) + lp[0]],
                                  dtype=torch.float64, device=dev)
                masks = torch.zeros(1, dtype=torch.int64, device=dev)
                w = torch.exp(sc - LOG_NORMALIZER)
            else:
                ranks = torch.arange(r, r + B, dtype=torch.int64, device=dev)
                idx = unrank_combinations(ranks, k, binom, torch)
                Z = Zxx[idx.unsqueeze(2), idx.unsqueeze(1)]
                b = Zxy[idx].unsqueeze(-1)
                L = torch.linalg.cholesky(Z)
                u = torch.linalg.solve_triangular(L, b, upper=False)
                ess = (u.squeeze(-1) ** 2).sum(dim=1)
                omr = torch.clamp((tss - ess) / tss_norm, min=tiny)
                sc = (0.5 * (df - k - k_alw) * log1pg
                      - 0.5 * df * torch.log1p(G * omr) + lp[k])
                masks = (torch.ones_like(idx) << idx).sum(dim=1)
                w = torch.exp(sc - LOG_NORMALIZER)   # exact global PMPs
                beta_hat = torch.linalg.solve_triangular(
                    L.transpose(1, 2), u, upper=True).squeeze(-1)
                zinv = torch.cholesky_inverse(L).diagonal(dim1=-2, dim2=-1)
                b_gam = tss_norm - s * (ess_always + ess)          # (B,)
                loc = s * beta_hat                                  # (B,k)
                scale = torch.sqrt((b_gam / df).unsqueeze(1) * s * zinv)
                x = grid[idx]                                       # (B,k,G)
                z = (x - loc.unsqueeze(-1)) / scale.unsqueeze(-1)
                logpdf = (c_t - torch.log(scale).unsqueeze(-1)
                          - 0.5 * (df + 1) * torch.log1p(z * z / df))
                contrib = w.unsqueeze(-1).unsqueeze(-1) * torch.exp(logpdf)
                onehot = torch.nn.functional.one_hot(idx, P).to(torch.float64)
                dens += torch.einsum('bkp,bkg->pg', onehot, contrib)
            sum_w += w.sum()
            cs = torch.cat([top_s, sc])
            cm_ = torch.cat([top_m, masks])
            best = torch.topk(cs, k=K_MAX)
            top_s, top_m = best.values, cm_[best.indices]
            done += B
            r += B
            now = time.time()
            if now - last_pr >= PROGRESS_EVERY_S and PROGRESS_EVERY_S:
                rate = done / max(now - t0, 1e-9)
                print(f'  pass2 [{done / N_MODELS:7.2%}] k={k:2d} '
                      f'{done:,}/{N_MODELS:,}  {rate:,.0f} models/s  '
                      f'ETA {(N_MODELS - done) / max(rate, 1e-9):,.0f} s',
                      flush=True)
                last_pr = now
            if now - last_ck >= CHECKPOINT_EVERY_S:
                save_ckpt(k if r < total_k else k + 1, r if r < total_k else 0)
                last_ck = now
    assert done == N_MODELS, f'pass2 incomplete: {done:,}'
    total_w = float(sum_w.cpu())
    assert abs(total_w - 1.0) < 1e-9, (
        f'sum of exact PMPs = {total_w!r}: saved log-normalizer inconsistent')
    order = torch.argsort(top_s, descending=True)
    scores = top_s[order].cpu().numpy()
    masks_np = top_m[order].cpu().numpy()
    keep = np.isfinite(scores)
    pmp = np.exp(scores[keep] - LOG_NORMALIZER)
    topk = pd.DataFrame({
        'model_rank': np.arange(1, keep.sum() + 1),
        'model_mask': masks_np[keep],
        'model_size': [bin(int(m)).count('1') for m in masks_np[keep]],
        'model_pmp_global': pmp,
        'cumulative_pmp_global': np.cumsum(pmp),
    })
    topk.to_parquet(TOPK_PARQUET, index=False)
    dens_np = dens.cpu().numpy()
    cond = dens_np / np.maximum(PIP, 1e-300)[:, None]   # conditional density
    np.savez(DENSITY_NPZ, grid=_pass2_grid(), weighted_density=dens_np,
             conditional_density=cond, pip=PIP, df=df,
             total_pmp_mass=total_w)
    save_ckpt(P + 1, 0)
    print(f'pass2 complete: {done:,} models, sum PMP = {total_w:.15f}, '
          f'{time.time() - t0:,.0f} s this session')

if TOPK_PARQUET.exists() and DENSITY_NPZ.exists():
    print('pass2 outputs already on Drive — skipping (idempotent)')
else:
    run_pass2()

TOPK = pd.read_parquet(TOPK_PARQUET)
DENS = np.load(DENSITY_NPZ)
print(f'top-{len(TOPK)} table loaded; cumulative PMP of top {len(TOPK)}: '
      f'{TOPK.cumulative_pmp_global.iloc[-1]:.6f}')


In [ ]:
# [PLOT-PMP] Exact PMP graph (bmagraph pmp equivalent, global PMPs)
def bma_pmp_plot(top_n=None, coverage=None, normalize_display=False):
    """top_n: show the N highest-PMP models; coverage: instead select the
    minimal K with cumulative global PMP >= coverage. PMPs are ALWAYS the
    exact global probabilities over all 2^p models; the omitted mass is an
    explicit 'Other models' bar. normalize_display=True (NON-DEFAULT,
    clearly labelled) rescales only the displayed bars for shape reading."""
    assert (top_n is None) != (coverage is None)
    if coverage is not None:
        K = int(np.searchsorted(TOPK.cumulative_pmp_global.values, coverage) + 1)
        assert K <= len(TOPK), (f'coverage {coverage} needs more than the '
                                f'{len(TOPK)} extracted models; raise K_MAX')
        sel_label = f'minimal K = {K} reaching coverage >= {coverage}'
    else:
        K = min(top_n, len(TOPK))
        sel_label = f'top {K} models'
    t = TOPK.head(K).copy()
    other = 1.0 - t.model_pmp_global.sum()
    shown = np.append(t.model_pmp_global.values, other)
    assert abs(shown.sum() - 1.0) < 1e-9, 'displayed + Other must sum to 1'
    disp = shown / shown[:-1].sum() if normalize_display else shown

    fig, ax = plt.subplots(figsize=(max(8, 0.16 * K + 2), 4.5))
    xs = np.arange(1, K + 1)
    ax.bar(xs, disp[:-1], color='#1f77b4',
           label='exact global PMP (denominator: all 2^%d models)' % P)
    ax.bar([K + 1.5], [disp[-1]], color='0.6', hatch='//',
           label=f'Other models (exact omitted mass = {other:.6f})')
    ax2 = ax.twinx()
    ax2.plot(xs, t.cumulative_pmp_global, color='#d62728', marker='.',
             ms=3, lw=1, label='cumulative global PMP')
    ax2.set_ylabel('cumulative global PMP'); ax2.set_ylim(0, 1.02)
    ax2.grid(False)
    ax.set_xlabel('model rank (descending global PMP)')
    ax.set_ylabel('displayed share (renormalized — NOT probabilities)'
                  if normalize_display else 'posterior model probability')
    title = f'Posterior model probabilities, p = {P} ({sel_label})'
    if normalize_display:
        title += '  [display-normalized mode — bars are not global PMPs]'
    ax.set_title(title)
    lines = [f'#{r.model_rank}: {{{", ".join(mask_vars(r.model_mask))}}}  '
             f'PMP={r.model_pmp_global:.4f}  cum={r.cumulative_pmp_global:.4f}'
             for r in t.head(5).itertuples()]
    ax.text(0.02, 0.97, '\n'.join(lines), transform=ax.transAxes, va='top',
            fontsize=7, family='monospace',
            bbox=dict(fc='white', alpha=0.8, ec='0.7'))
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='center right', fontsize=8)
    t['included_variables'] = t.model_mask.map(
        lambda m: ' '.join(mask_vars(m)))
    t['other_model_mass'] = other
    return fig, t

for N in (20, 50, 100):
    fig, tbl = bma_pmp_plot(top_n=N)
    save_fig(fig, f'pmp_top{N}')
    save_table(tbl, f'pmp_top{N}')
    plt.show() if N == 50 else plt.close(fig)


In [ ]:
# [PLOT-MSIZE] Model-size distribution (bmagraph msize equivalent)
sizes = np.arange(P + 1)
prior_sizes = np.array([math.comb(P, k) * math.exp(LOG_PRIOR(k))
                        for k in sizes])   # beta-binomial(1,1): uniform
assert abs(prior_sizes.sum() - 1.0) < 1e-12
assert abs(SIZE_DIST.sum() - 1.0) < 1e-9
prior_mean = float(sizes @ prior_sizes)
post_mean = float(payload['mean_model_size'])

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(sizes, prior_sizes, 'o--', color='0.45', label='prior (analytical)')
ax.plot(sizes, SIZE_DIST, 'o-', color='#1f77b4', label='posterior (exact)')
ax.axvline(prior_mean, color='0.45', ls=':',
           label=f'prior mean = {prior_mean:.2f}')
ax.axvline(post_mean, color='#d62728', ls=':',
           label=f'posterior mean = {post_mean:.3f}')
ax.set_xticks(sizes)
ax.set_xlabel('model size (number of included optional predictors)')
ax.set_ylabel('probability')
ax.set_title(f'Model-size distribution, p = {P} '
             f'(exact posterior over all 2^{P} models; '
             'beta-binomial(1,1) prior)')
ax.legend(fontsize=8)
save_fig(fig, 'msize')
save_table(pd.DataFrame({'model_size': sizes, 'prior': prior_sizes,
                         'posterior': SIZE_DIST}), 'msize_distribution')
plt.show()


In [ ]:
# [PLOT-VARMAP] Coverage-based variable-inclusion map
# bmagraph varmap equivalent, improved selection:
# minimal K reaching the posterior-coverage target

PMP_COVERAGE_TARGET = 0.90  # also supported: 0.80, 0.95, 0.99

# ------------------------------------------------------------------
# Select the minimum number of top models needed to reach coverage
# ------------------------------------------------------------------
cum = TOPK.cumulative_pmp_global.values

K_COV = int(np.searchsorted(cum, PMP_COVERAGE_TARGET) + 1)

assert K_COV <= len(TOPK), (
    f'target {PMP_COVERAGE_TARGET} needs more than '
    f'{len(TOPK)} models; raise K_MAX in [PASS2]'
)
assert cum[K_COV - 1] >= PMP_COVERAGE_TARGET
assert K_COV == 1 or cum[K_COV - 2] < PMP_COVERAGE_TARGET, (
    'K must be minimal'
)

sel = TOPK.head(K_COV)
CPMP_K = float(sel.model_pmp_global.sum())
OTHER_MASS = float(np.clip(1.0 - CPMP_K, 0.0, 1.0))

print(
    f'coverage target {PMP_COVERAGE_TARGET}: minimal K = {K_COV}, '
    f'achieved CPMP = {CPMP_K:.6f}, '
    f'omitted models = {N_MODELS - K_COV:,}, '
    f'exact omitted mass = {OTHER_MASS:.6f}'
)

# ------------------------------------------------------------------
# Conditional coefficients for the selected K models
# ------------------------------------------------------------------
from scipy.linalg import cho_factor, cho_solve

Zxx_np = X_r.T @ X_r
Zxy_np = X_r.T @ y_r
S_G = G / (1.0 + G)
COEF_ZERO_TOL = 1e-12

cond_coef = np.zeros((K_COV, P), dtype=float)
incl = np.zeros((K_COV, P), dtype=bool)

for i, m in enumerate(sel.model_mask):
    jj = [j for j in range(P) if (int(m) >> j) & 1]
    incl[i, jj] = True

    if jj:
        c_, lo_ = cho_factor(
            Zxx_np[np.ix_(jj, jj)],
            lower=True,
        )

        cond_coef[i, jj] = S_G * cho_solve(
            (c_, lo_),
            Zxy_np[jj],
        )

# All predictors, ordered from highest to lowest exact PIP
row_order = np.argsort(-PIP)

# ------------------------------------------------------------------
# Figure function
# ------------------------------------------------------------------
def varmap_figure(
    proportional=True,
    max_cols=None,
    name='varmap',
):
    cols = min(K_COV, max_cols) if max_cols else K_COV

    if proportional:
        widths = sel.model_pmp_global.values[:cols]
    else:
        widths = np.full(cols, CPMP_K / K_COV)

    xedges = np.concatenate(([0.0], np.cumsum(widths)))

    # When truncated, this block also contains selected models not shown.
    other_w = (
        OTHER_MASS
        if cols == K_COV
        else float(np.clip(1.0 - widths.sum(), 0.0, 1.0))
    )

    fig, (ax, axp) = plt.subplots(
        1,
        2,
        figsize=(
            max(10, min(0.09 * cols + 4, 60)),
            0.30 * P + 2.5,
        ),
        gridspec_kw={'width_ratios': [5, 1]},
        sharey=True,
    )

    colors = {
        1: '#2166ac',   # positive conditional coefficient
        -1: '#b2182b',  # negative conditional coefficient
        0: '#ffee99',   # numerically zero coefficient
    }

    # --------------------------------------------------------------
    # Main variable-inclusion map
    # --------------------------------------------------------------
    for ri, j in enumerate(row_order):
        for ci in range(cols):
            if not incl[ci, j]:
                continue

            v = cond_coef[ci, j]

            sign = (
                0
                if abs(v) < COEF_ZERO_TOL
                else (1 if v > 0 else -1)
            )

            ax.add_patch(
                plt.Rectangle(
                    (xedges[ci], ri),
                    widths[ci],
                    1,
                    facecolor=colors[sign],
                    edgecolor='none',
                )
            )

    # Neutral block for all models not displayed individually
    ax.add_patch(
        plt.Rectangle(
            (xedges[cols], -0.5),
            other_w,
            P + 1,
            facecolor='0.85',
            edgecolor='0.5',
            hatch='///',
        )
    )

    if cols == K_COV:
        other_label = (
            'Other models\n'
            f'exact mass = {OTHER_MASS:.4f}\n'
            f'{N_MODELS - K_COV:,} models omitted'
        )
    else:
        undisplayed_mass = float(
            np.clip(1.0 - widths.sum(), 0.0, 1.0)
        )

        other_label = (
            'Models not shown\n'
            f'total mass = {undisplayed_mass:.4f}\n'
            f'{N_MODELS - cols:,} models'
        )

    ax.text(
        xedges[cols] + other_w / 2,
        P / 2,
        other_label,
        rotation=90,
        ha='center',
        va='center',
        fontsize=8,
    )

    # Thin separators between displayed model columns
    for edge in xedges:
        ax.axvline(
            edge,
            color='white',
            linewidth=0.3,
        )

    ax.set_yticks(np.arange(P) + 0.5)

    ax.set_yticklabels(
        [predictors[j] for j in row_order],
        fontsize=8,
    )

    ax.set_ylim(P, -0.5)
    ax.set_xlim(0.0, xedges[cols] + other_w)

    if proportional:
        ax.set_xlabel(
            'cumulative global PMP '
            '(column width proportional to exact global PMP)'
        )
    else:
        ax.set_xlabel(
            f'{cols} models shown at equal width '
            '(widths NOT proportional to PMP); '
            'x-axis not to probability scale'
        )

    trunc = (
        f', first {cols} of K = {K_COV} columns (TRUNCATED)'
        if cols < K_COV
        else ''
    )

    ax.set_title(
        f'Variable-inclusion map, p = {P}: '
        f'minimal K = {K_COV} models with '
        f'cumulative global PMP {CPMP_K:.4f} '
        f'>= target {PMP_COVERAGE_TARGET}'
        f'{trunc}\n'
        'blue = included, positive conditional coefficient; '
        'red = included, negative; '
        'pale yellow = |coef| < 1e-12; '
        'white = excluded'
    )

    # --------------------------------------------------------------
    # Right panel: exact posterior inclusion probabilities
    # --------------------------------------------------------------
    y_positions = np.arange(P) + 0.5

    axp.barh(
        y_positions,
        PIP[row_order],
        color='#1f77b4',
        height=0.8,
    )

    # Extra horizontal space keeps all labels outside the bars.
    PIP_XMAX = 1.12
    PIP_LABEL_PAD = 0.015

    axp.set_xlim(0.0, PIP_XMAX)
    axp.set_xlabel('PIP (exact)')

    for ri, j in enumerate(row_order):
        pip_j = float(np.clip(PIP[j], 0.0, 1.0))

        axp.text(
            pip_j + PIP_LABEL_PAD,
            ri + 0.5,
            f'{pip_j:.2f}',
            va='center',
            ha='left',
            fontsize=7,
            clip_on=False,
        )

    save_fig(fig, name)

    return fig


# ------------------------------------------------------------------
# Main proportional-width figure
# ------------------------------------------------------------------
fig_full = varmap_figure(
    proportional=True,
    name='varmap_proportional',
)

plt.show()


# ------------------------------------------------------------------
# Equal-width alternative
# ------------------------------------------------------------------
fig_eq = varmap_figure(
    proportional=False,
    name='varmap_equal_width',
)

plt.close(fig_eq)


# ------------------------------------------------------------------
# Optional compact figure if many models are needed for coverage
# ------------------------------------------------------------------
if K_COV > 60:
    fig_c = varmap_figure(
        proportional=True,
        max_cols=60,
        name='varmap_compact_truncated',
    )

    plt.close(fig_c)


# ------------------------------------------------------------------
# Complete underlying data table
# K_COV selected models x all P predictors, with no predictor cutoff
# ------------------------------------------------------------------
rows = []

for ci, r in enumerate(sel.itertuples()):
    for j in range(P):
        v = cond_coef[ci, j]

        if not incl[ci, j]:
            conditional_coefficient = np.nan
            coefficient_sign = 0

        elif abs(v) < COEF_ZERO_TOL:
            conditional_coefficient = float(v)
            coefficient_sign = 0

        else:
            conditional_coefficient = float(v)
            coefficient_sign = 1 if v > 0 else -1

        rows.append(
            {
                'predictor': predictors[j],
                'predictor_index': j,
                'pip': float(PIP[j]),
                'model_rank': int(r.model_rank),
                'model_mask': int(r.model_mask),
                'model_size': int(r.model_size),
                'model_pmp_global': float(r.model_pmp_global),
                'cumulative_pmp_global': float(
                    r.cumulative_pmp_global
                ),
                'included': bool(incl[ci, j]),
                'conditional_coefficient': conditional_coefficient,
                'coefficient_sign': coefficient_sign,
                'selected_for_coverage': True,
                'coverage_target': PMP_COVERAGE_TARGET,
                'other_model_mass': OTHER_MASS,
            }
        )

varmap_tbl = pd.DataFrame(rows)

assert varmap_tbl.predictor.nunique() == P, (
    'all predictors must appear'
)

save_table(
    varmap_tbl,
    f'varmap_coverage{int(PMP_COVERAGE_TARGET * 100)}',
)

In [ ]:
# [PLOT-COEFDENSITY] Exact coefficient densities
#
# Blue:
#   exact posterior density conditional on predictor inclusion.
#
# Red spike:
#   posterior point mass at beta = 0, equal to 1 - PIP.

GRID = DENS['grid']                    # shape: (P, G)
COND = DENS['conditional_density']     # shape: (P, G), integrates to 1

SHOW_ALL_30 = False
n_show = P if SHOW_ALL_30 else min(10, P)
show = row_order[:n_show]

ZERO_MASS_EPS = 1e-6

_trapz = getattr(np, 'trapezoid', None) or np.trapz

# Validate conditional densities.
for j in show:
    integral = _trapz(COND[j], GRID[j])
    assert abs(integral - 1.0) < 5e-3, (
        f'Conditional density for {predictors[j]} '
        f'integrates to {integral:.8f}, not 1.'
    )

ncols = 5
nrows = -(-n_show // ncols)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(3.35 * ncols, 2.9 * nrows),
    squeeze=False,
)

for ax_i, j in enumerate(show):
    row = ax_i // ncols
    col = ax_i % ncols

    ax = axes[row][col]

    pip_j = float(np.clip(PIP[j], 0.0, 1.0))
    zero_mass = float(np.clip(1.0 - pip_j, 0.0, 1.0))

    # ----------------------------------------------------------
    # Primary axis: density conditional on inclusion
    # ----------------------------------------------------------
    ax.plot(
        GRID[j],
        COND[j],
        color='#1f77b4',
        linewidth=1.35,
    )

    ax.fill_between(
        GRID[j],
        COND[j],
        color='#1f77b4',
        alpha=0.22,
    )

    ax.set_ylabel(
        'Density | included' if col == 0 else '',
        fontsize=8,
    )

    ax.tick_params(
        axis='both',
        labelsize=7,
    )

    # Neutral reference at beta = 0.
    ax.axvline(
        0.0,
        color='0.72',
        linewidth=0.8,
        linestyle=':',
        zorder=1,
    )

    # ----------------------------------------------------------
    # Secondary axis: point mass at beta = 0
    # ----------------------------------------------------------
    ax2 = ax.twinx()
    ax2.set_ylim(0.0, 1.0)

    is_last_visible_column = (
        col == ncols - 1
        or ax_i == n_show - 1
    )

    if is_last_visible_column:
        ax2.set_ylabel(
            r'$P(\beta=0)$',
            fontsize=8,
            color='#d62728',
        )
        ax2.tick_params(
            axis='y',
            labelsize=7,
            colors='#d62728',
        )
    else:
        ax2.set_ylabel('')
        ax2.tick_params(
            axis='y',
            right=False,
            labelright=False,
        )

    if zero_mass > ZERO_MASS_EPS:
        # Red spike whose height is exactly 1 - PIP.
        ax2.vlines(
            x=0.0,
            ymin=0.0,
            ymax=zero_mass,
            color='#d62728',
            linewidth=2.0,
            zorder=5,
        )

        ax2.plot(
            [0.0],
            [zero_mass],
            marker='v',
            markersize=5.5,
            color='#d62728',
            zorder=6,
        )

        # Compact numeric label beside the red spike.
        ax2.annotate(
            f'{zero_mass:.2f}',
            xy=(0.0, zero_mass),
            xytext=(5, -2),
            textcoords='offset points',
            color='#d62728',
            fontsize=7,
            fontweight='bold',
            ha='left',
            va='top',
            zorder=7,
            bbox={
                'facecolor': 'white',
                'edgecolor': 'none',
                'alpha': 0.75,
                'pad': 0.8,
            },
        )

    # Clean panel heading.
    ax.set_title(
        f'{predictors[j]}   PIP = {pip_j:.2f}',
        fontsize=9,
    )

# Hide unused panels.
for ax_i in range(n_show, nrows * ncols):
    axes[ax_i // ncols][ax_i % ncols].axis('off')

# Common explanation replaces repeated panel legends.
fig.suptitle(
    f'Posterior coefficient densities, p = {P}\n'
    'Blue: exact density conditional on inclusion; '
    r'red spike: $P(\beta=0)=1-\mathrm{PIP}$',
    y=1.015,
    fontsize=13,
)

fig.tight_layout()

save_fig(
    fig,
    f'coefdensity_top{n_show}',
)

plt.show()

# --------------------------------------------------------------
# Save complete plotting data for all predictors
# --------------------------------------------------------------
dens_tbl = pd.concat(
    [
        pd.DataFrame(
            {
                'predictor': predictors[j],
                'predictor_index': j,
                'x': GRID[j],
                'conditional_density': COND[j],
                'unconditional_continuous_part': (
                    float(np.clip(PIP[j], 0.0, 1.0))
                    * COND[j]
                ),
                'pip': float(np.clip(PIP[j], 0.0, 1.0)),
                'zero_point_mass': float(
                    np.clip(1.0 - PIP[j], 0.0, 1.0)
                ),
                'zero_mass_drawn': bool(
                    np.clip(1.0 - PIP[j], 0.0, 1.0)
                    > ZERO_MASS_EPS
                ),
                'zero_mass_draw_threshold': ZERO_MASS_EPS,
            }
        )
        for j in range(P)
    ],
    ignore_index=True,
)

save_table(
    dens_tbl,
    'coefdensity_grids',
)

In [ ]:
# [PLOT-COEFDENSITY-SLIDES] Exact coefficient densities, one figure per predictor
#
# One slide-ready figure per predictor.
# Blue:
#   exact posterior density conditional on predictor inclusion.
# Red spike:
#   posterior point mass at beta = 0, equal to 1 - PIP.

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GRID = DENS['grid']                    # shape: (P, G)
COND = DENS['conditional_density']     # shape: (P, G), integrates to 1

SHOW_ALL_30 = False
n_show = P if SHOW_ALL_30 else min(10, P)
show = row_order[:n_show]

ZERO_MASS_EPS = 1e-6
_trapz = getattr(np, 'trapezoid', None) or np.trapz

# ------------------------------------------------------------------
# Helpers
# ------------------------------------------------------------------
def _safe_name(x):
    x = str(x)
    x = re.sub(r'[^A-Za-z0-9._-]+', '_', x)
    return x.strip('_')

# Validate conditional densities for selected predictors.
for j in show:
    integral = _trapz(COND[j], GRID[j])
    assert abs(integral - 1.0) < 5e-3, (
        f'Conditional density for {predictors[j]} '
        f'integrates to {integral:.8f}, not 1.'
    )

# ------------------------------------------------------------------
# Slide-style plotting parameters
# ------------------------------------------------------------------
SLIDE_FIGSIZE = (13.333, 7.5)   # 16:9 aspect, good for slides
TITLE_FS = 24
SUBTITLE_FS = 14
AXIS_LABEL_FS = 16
TICK_FS = 13
ANNOT_FS = 15
LINE_FS = 2.4
SPIKE_LW = 4.0

# ------------------------------------------------------------------
# Plot one figure per predictor
# ------------------------------------------------------------------
for rank, j in enumerate(show, start=1):
    pip_j = float(np.clip(PIP[j], 0.0, 1.0))
    zero_mass = float(np.clip(1.0 - pip_j, 0.0, 1.0))

    fig, ax = plt.subplots(figsize=SLIDE_FIGSIZE)

    # --------------------------------------------------------------
    # Primary axis: density conditional on inclusion
    # --------------------------------------------------------------
    ax.plot(
        GRID[j],
        COND[j],
        color='#1f77b4',
        linewidth=LINE_FS,
        zorder=3,
    )

    ax.fill_between(
        GRID[j],
        COND[j],
        color='#1f77b4',
        alpha=0.20,
        zorder=2,
    )

    ax.axvline(
        0.0,
        color='0.70',
        linewidth=1.2,
        linestyle=':',
        zorder=1,
    )

    ax.set_xlabel(r'Coefficient value $\beta$', fontsize=AXIS_LABEL_FS)
    ax.set_ylabel('Density | included', fontsize=AXIS_LABEL_FS)
    ax.tick_params(axis='both', labelsize=TICK_FS)

    # Light grid for slide readability
    ax.grid(axis='y', alpha=0.20, linewidth=0.8)

    # --------------------------------------------------------------
    # Secondary axis: point mass at beta = 0
    # --------------------------------------------------------------
    ax2 = ax.twinx()
    ax2.set_ylim(0.0, 1.0)
    ax2.set_ylabel(r'$P(\beta=0)$', fontsize=AXIS_LABEL_FS, color='#d62728')
    ax2.tick_params(axis='y', labelsize=TICK_FS, colors='#d62728')

    if zero_mass > ZERO_MASS_EPS:
        ax2.vlines(
            x=0.0,
            ymin=0.0,
            ymax=zero_mass,
            color='#d62728',
            linewidth=SPIKE_LW,
            zorder=5,
        )

        ax2.plot(
            [0.0],
            [zero_mass],
            marker='v',
            markersize=9,
            color='#d62728',
            zorder=6,
        )

        ax2.annotate(
            f'{zero_mass:.2f}',
            xy=(0.0, zero_mass),
            xytext=(10, -4),
            textcoords='offset points',
            color='#d62728',
            fontsize=ANNOT_FS,
            fontweight='bold',
            ha='left',
            va='top',
            zorder=7,
            bbox={
                'facecolor': 'white',
                'edgecolor': 'none',
                'alpha': 0.85,
                'pad': 1.2,
            },
        )

    # --------------------------------------------------------------
    # Titles and annotations
    # --------------------------------------------------------------
    fig.suptitle(
        f'{predictors[j]}',
        fontsize=TITLE_FS,
        y=0.97,
    )

    fig.text(
        0.5,
        0.915,
        f'Posterior coefficient density | PIP = {pip_j:.2f} | '
        r'$P(\beta=0)$' + f' = {zero_mass:.2f}',
        ha='center',
        va='center',
        fontsize=SUBTITLE_FS,
    )

    # Optional slide footer
    fig.text(
        0.01,
        0.015,
        f'Rank by PIP: {rank} of {n_show} shown | p = {P}',
        ha='left',
        va='bottom',
        fontsize=11,
        color='0.35',
    )

    fig.tight_layout(rect=[0.03, 0.05, 0.97, 0.88])

    # Save each figure individually
    fname = f'coefdensity_slide_{rank:02d}_{_safe_name(predictors[j])}'
    save_fig(fig, fname)

    plt.show()
    plt.close(fig)

# ------------------------------------------------------------------
# Save complete plotting data for all predictors
# ------------------------------------------------------------------
dens_tbl = pd.concat(
    [
        pd.DataFrame(
            {
                'predictor': predictors[j],
                'predictor_index': j,
                'x': GRID[j],
                'conditional_density': COND[j],
                'unconditional_continuous_part': (
                    float(np.clip(PIP[j], 0.0, 1.0)) * COND[j]
                ),
                'pip': float(np.clip(PIP[j], 0.0, 1.0)),
                'zero_point_mass': float(np.clip(1.0 - PIP[j], 0.0, 1.0)),
                'zero_mass_drawn': bool(
                    np.clip(1.0 - PIP[j], 0.0, 1.0) > ZERO_MASS_EPS
                ),
                'zero_mass_draw_threshold': ZERO_MASS_EPS,
            }
        )
        for j in range(P)
    ],
    ignore_index=True,
)

save_table(
    dens_tbl,
    'coefdensity_grids',
)

In [ ]:
# [PLOT-PIP] Posterior inclusion probabilities (all predictors, exact)

order = np.argsort(PIP)   # ascending for horizontal bars, top = highest
fig, ax = plt.subplots(figsize=(7, 0.28 * P + 1.5))

ax.barh(np.arange(P), PIP[order], color='#1f77b4')

ax.axvline(
    PRIOR_PIP,
    color='0.4',
    ls='--',
    label=f'prior inclusion probability = {PRIOR_PIP:.2f} '
          '(beta-binomial(1,1))'
)

ax.set_yticks(np.arange(P))
ax.set_yticklabels([predictors[j] for j in order], fontsize=8)

# Put labels outside the bars, to the right
label_pad = 0.008
xmax = 1.10

for i, j in enumerate(order):
    x_text = min(PIP[j] + label_pad, xmax - 0.04)
    ax.text(
        x_text,
        i,
        f'{PIP[j]:.2f}',
        va='center',
        ha='left',
        fontsize=8
    )

ax.set_xlim(0, xmax)
ax.set_xlabel(f'posterior inclusion probability (exact, all 2^{P} models)')
ax.set_title(f'Posterior inclusion probabilities, p = {P}')
ax.legend(fontsize=8, loc='lower right')

save_fig(fig, 'pip')

save_table(
    pd.DataFrame({
        'predictor': predictors,
        'pip': PIP,
        'posterior_mean': COEF_MEAN,
        'posterior_sd': COEF_SD,
        'prior_pip': PRIOR_PIP
    }).sort_values('pip', ascending=False),
    'pip_table'
)

plt.show()

Reopening after a disconnect: run all cells again — the enumeration resumes
from the newest Drive checkpoint (at most ~60 s of work lost) or is skipped
once `results.json` exists; `[PASS2]` likewise resumes or skips; the plot
cells always rebuild from the saved Drive artifacts.
